# 🚀 Notebook Chạy Demo Dịch Thuật Việt - Anh với Mini-Transformer
Notebook này được thiết kế để giúp bạn **tải mô hình đã được huấn luyện thành công trên Kaggle** (file checkpoint `.pt`) và chạy dịch thử nghiệm (Inference) các câu tiếng Việt sang tiếng Anh một cách nhanh chóng.

### 🌟 Các chức năng chính:
1. **Khởi tạo nhanh BPETokenizer** đồng bộ với dữ liệu đã train.
2. **Định nghĩa kiến trúc Mini-Transformer** chuẩn PyTorch.
3. **Tải trọng số đã lưu (Weights)** từ checkpoint `.pt` của bạn.
4. **Thuật toán giải mã Beam Search** cải tiến với Length Penalty.
5. **Giao diện Demo Gradio trực quan**: Nhập câu tiếng Việt và nhận bản dịch tiếng Anh ngay lập tức trên giao diện Web UI mượt mà.

---
### 🛠️ Hướng dẫn kết nối file trọng số trên Kaggle:
- Nếu bạn chạy trên chính Notebook đã train trên Kaggle: Các file output `.pt` nằm ở thư mục `/kaggle/working/`.
- Nếu bạn mở một Notebook mới trên Kaggle: Nhìn sang góc phải màn hình -> Click **+ Add Data** -> Chọn tab **Notebooks** -> Tìm tên Notebook chứa phiên bản đã chạy trước đó của bạn -> Bấm nút **Add**. File `.pt` lúc này sẽ nằm tại: `'/kaggle/input/<tên-notebook-của-bạn>/mini_transformer_best_bleu.pt'`.


## Bước 1: Cài đặt các thư viện cần thiết
Chúng ta cần cài đặt `datasets` (để tải tập dữ liệu huấn luyện nhanh Tokenizer), `tokenizers` (thuật toán BPE cực nhanh) và `gradio` (tạo giao diện Demo).


In [ ]:
!pip install -q datasets tokenizers gradio


## Bước 2: Khai báo thư viện và kiểm tra thiết bị (Device)


In [ ]:
import torch
import torch.nn as nn
import math
import re
import html
from datasets import load_dataset
from tokenizers import Tokenizer, normalizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKC, Lowercase

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng để chạy Demo: {device}")


## Bước 3: Định nghĩa và Khởi tạo BPETokenizer
Để dịch chính xác, **Tokenizer bắt buộc phải có bộ từ vựng và ID giống hệt lúc huấn luyện mô hình**.

### 👉 Có 2 cách để khởi tạo:
1. **Cách 1 (Khuyên dùng - Chính xác 100%):** Tải file `tokenizer.json` (được lưu sau khi chạy huấn luyện) và load trực tiếp.
2. **Cách 2 (Chạy thử/Tạm thời):** Tự động tải dataset và huấn luyện lại tokenizer từ đầu. *Lưu ý: BPE training có tính ngẫu nhiên nhẹ khi chạy đa luồng, do đó có thể gây lỗi lệch ID (mô hình dịch ra từ vô nghĩa).* Nếu gặp tình trạng này, bạn bắt buộc phải lưu file `tokenizer.json` từ phiên bản chạy huấn luyện.


In [ ]:
import os
def clean_text(text):
    text = html.unescape(text)  # Giải mã thực thể HTML: &apos; -> ', &quot; -> "
    text = text.strip().lower()
    # Giữ lại các dấu câu cơ bản để hỗ trợ dịch chính xác hơn, chỉ chuẩn hóa unicode và dấu ngoặc
    text = re.sub(r"[“”“«»]", "\"", text)
    text = re.sub(r"[‘’`´]", "'", text)
    text = re.sub(r"[–—]", "-", text)
    return text

class BPETokenizer:
    def __init__(self, vocab_size=18000, model_path=None):
        self.vocab_size = vocab_size
        if model_path and os.path.exists(model_path):
            self.tokenizer = Tokenizer.from_file(model_path)
            print(f"🎉 Đã load thành công tokenizer từ file: {model_path}")
        else:
            if model_path:
                print(f"⚠️ Không tìm thấy file {model_path}. Tiến hành huấn luyện động tokenizer (có thể gây lệch ID!).")
            self.tokenizer = Tokenizer(BPE(unk_token="<unk>"))
            self.tokenizer.normalizer = normalizers.Sequence([NFKC(), Lowercase()])
            self.tokenizer.pre_tokenizer = Whitespace()
            self.trainer = BpeTrainer(
                special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"],
                vocab_size=self.vocab_size
            )

    def train(self, sentences):
        cleaned = [clean_text(s) for s in sentences]
        self.tokenizer.train_from_iterator(cleaned, self.trainer)

    def encode(self, sentence, add_special=True):
        cleaned = clean_text(sentence)
        ids = self.tokenizer.encode(cleaned).ids
        if add_special:
            ids = [1] + ids + [2]  # <sos>=1, <eos>=2
        return ids

    def decode(self, ids):
        filtered = [idx for idx in ids if idx not in [0, 1, 2]]
        return self.tokenizer.decode(filtered)

    @property
    def word2idx(self):
        return {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}

    @property
    def idx2word(self):
        class _IdToToken:
            def __init__(self, tok):
                self.tok = tok
            def __getitem__(self, idx):
                val = self.tok.id_to_token(idx)
                return val if val is not None else "<unk>"
            def get(self, idx, default="<unk>"):
                val = self.tok.id_to_token(idx)
                return val if val is not None else default
        return _IdToToken(self.tokenizer)

    def __len__(self):
        return self.tokenizer.get_vocab_size()

# ĐƯỜNG DẪN TỚI FILE TOKENIZER CỦA BẠN (nếu có)
# Ví dụ: 'tokenizer.json' hoặc '/kaggle/input/.../tokenizer.json'
tokenizer_path = 'tokenizer.json'

if os.path.exists(tokenizer_path):
    shared_vocab = BPETokenizer(model_path=tokenizer_path)
else:
    # Cách 2: Tải dataset và train động lại
    print("👉 Không tìm thấy file tokenizer.json. Tiến hành huấn luyện lại tokenizer từ đầu...")
    print("Đang tải bộ dữ liệu IWSLT 2015 để huấn luyện Tokenizer...")
    raw_dataset = load_dataset("thainq107/iwslt2015-en-vi", trust_remote_code=True)
    vi_train_sentences = [item['vi'] for item in raw_dataset['train']]
    en_train_sentences = [item['en'] for item in raw_dataset['train']]
    shared_vocab = BPETokenizer(vocab_size=18000)
    shared_vocab.train(vi_train_sentences + en_train_sentences)
    print(f"Huấn luyện xong! Kích thước từ vựng: {len(shared_vocab)}")


## Bước 4: Khai báo Kiến trúc mô hình Mini-Transformer
Để PyTorch có thể load được tệp trọng số `.pt`, chúng ta cần định nghĩa lại các lớp cấu tạo nên mô hình khớp hoàn toàn với kiến trúc lúc huấn luyện.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model phải chia hết cho num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e4)
        attn_weights = self.dropout(torch.softmax(scores, dim=-1))
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(context)
        return output, attn_weights

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.w_2(self.dropout(self.relu(self.w_1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        norm_x = self.norm1(x)
        attn_out, attn_weights = self.self_attn(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out)
        norm_x = self.norm2(x)
        ff_out = self.feed_forward(norm_x)
        x = x + self.dropout(ff_out)
        return x, attn_weights

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        norm_x = self.norm1(x)
        self_attn_out, self_attn_weights = self.self_attn(norm_x, norm_x, norm_x, tgt_mask)
        x = x + self.dropout(self_attn_out)
        norm_x = self.norm2(x)
        cross_attn_out, cross_attn_weights = self.cross_attn(norm_x, enc_output, enc_output, src_mask)
        x = x + self.dropout(cross_attn_out)
        norm_x = self.norm3(x)
        ff_out = self.feed_forward(norm_x)
        x = x + self.dropout(ff_out)
        return x, self_attn_weights, cross_attn_weights

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len=5000, dropout=0.1, embedding=None):
        super(Encoder, self).__init__()
        self.d_model = d_model
        if embedding is not None:
            self.embedding = embedding
        else:
            self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        attn_list = []
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attn_list.append(attn_weights)
        return self.norm(x), attn_list

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len=5000, dropout=0.1, embedding=None):
        super(Decoder, self).__init__()
        self.d_model = d_model
        if embedding is not None:
            self.embedding = embedding
        else:
            self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        self_attn_list = []
        cross_attn_list = []
        for layer in self.layers:
            x, self_attn, cross_attn = layer(x, enc_output, src_mask, tgt_mask)
            self_attn_list.append(self_attn)
            cross_attn_list.append(cross_attn)
        return self.norm(x), self_attn_list, cross_attn_list

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_layers=6, num_heads=8, d_ff=2048, max_len=5000, dropout=0.1):
        super(Transformer, self).__init__()
        self.num_layers = num_layers
        
        if src_vocab_size == tgt_vocab_size:
            self.shared_embedding = nn.Embedding(src_vocab_size, d_model)
            self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout, embedding=self.shared_embedding)
            self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout, embedding=self.shared_embedding)
            self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
            self.generator.weight = self.shared_embedding.weight
        else:
            self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
            self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
            self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
            self.generator.weight = self.decoder.embedding.weight
        
        self.apply(self._init_weights)
        
        if src_vocab_size == tgt_vocab_size:
            self.generator.weight = self.shared_embedding.weight
        else:
            self.generator.weight = self.decoder.embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
        
        if isinstance(module, EncoderLayer) or isinstance(module, DecoderLayer):
            if hasattr(module, 'self_attn') and hasattr(module.self_attn, 'W_o'):
                nn.init.normal_(module.self_attn.W_o.weight, mean=0.0, std=0.02 / math.sqrt(2 * self.num_layers))
            if hasattr(module, 'cross_attn') and hasattr(module.cross_attn, 'W_o'):
                nn.init.normal_(module.cross_attn.W_o.weight, mean=0.0, std=0.02 / math.sqrt(2 * self.num_layers))
            if hasattr(module, 'feed_forward') and hasattr(module.feed_forward, 'w_2'):
                nn.init.normal_(module.feed_forward.w_2.weight, mean=0.0, std=0.02 / math.sqrt(2 * self.num_layers))

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output, enc_attn = self.encoder(src, src_mask)
        dec_output, dec_self_attn, dec_cross_attn = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        logits = self.generator(dec_output)
        return logits, enc_attn, dec_self_attn, dec_cross_attn


## Bước 5: Các hàm hỗ trợ Tạo Mask & Giải mã (Decoding)
Định nghĩa thuật toán tạo mặt nạ đệm và thuật toán **Beam Search** với Length Penalty nhằm tối ưu chất lượng câu dịch đầu ra.


In [ ]:
def make_src_mask(src, pad_idx=0):
    src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
    return src_mask.to(src.device)

def make_tgt_mask(tgt, pad_idx=0):
    tgt_pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
    seq_len = tgt.size(1)
    no_peak_mask = torch.tril(torch.ones((seq_len, seq_len), device=tgt.device)).bool()
    no_peak_mask = no_peak_mask.unsqueeze(0).unsqueeze(1)
    tgt_mask = tgt_pad_mask & no_peak_mask
    return tgt_mask.to(tgt.device)

def beam_search_decode(model, src_sentence, src_vocab, tgt_vocab, beam_size=5, max_len=50, alpha=0.6):
    model.eval()
    raw_model = model.module if hasattr(model, 'module') else model
    
    src_tokens = src_vocab.encode(src_sentence, add_special=False)
    src_tensor = torch.tensor([src_tokens], dtype=torch.long).to(device)
    src_mask = make_src_mask(src_tensor)
    
    with torch.no_grad():
        enc_output, _ = raw_model.encoder(src_tensor, src_mask)
        # Cấu trúc mỗi beam: (danh_sách_tokens, điểm_log_prob_lũy_kế)
        beams = [([tgt_vocab.word2idx["<sos>"]], 0.0)]
        
        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[-1] == tgt_vocab.word2idx["<eos>"]:
                    candidates.append((seq, score))
                    continue
                
                tgt_tensor = torch.tensor([seq], dtype=torch.long).to(device)
                tgt_mask = make_tgt_mask(tgt_tensor)
                dec_output, _, _ = raw_model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
                logits = raw_model.generator(dec_output)
                log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)
                
                top_log_probs, top_ids = torch.topk(log_probs, beam_size)
                for i in range(beam_size):
                    next_token = top_ids[i].item()
                    prob = top_log_probs[i].item()
                    candidates.append((seq + [next_token], score + prob))
            
            # Chuẩn hóa điểm dựa trên độ dài chuỗi (Length Penalty)
            def get_norm_score(cand):
                s_seq, s_score = cand
                length = len(s_seq) - 1  # Bỏ qua token <sos>
                if length <= 0:
                    return s_score
                lp = ((5 + length) ** alpha) / (6 ** alpha)
                return s_score / lp
            
            candidates = sorted(candidates, key=get_norm_score, reverse=True)
            beams = candidates[:beam_size]
            if all(seq[-1] == tgt_vocab.word2idx["<eos>"] for seq, _ in beams):
                break
        
        best_seq, _ = beams[0]
        decoded_translation = tgt_vocab.decode(best_seq)
        return decoded_translation

def translate(model, src_sentence, src_vocab, tgt_vocab, mode='beam', beam_size=5, max_len=50):
    if mode == 'beam':
        return beam_search_decode(model, src_sentence, src_vocab, tgt_vocab, beam_size=beam_size, max_len=max_len)
    
    # Giải mã Greedy Search (nếu không dùng beam)
    model.eval()
    raw_model = model.module if hasattr(model, 'module') else model
    src_tokens = src_vocab.encode(src_sentence, add_special=False)
    src_tensor = torch.tensor([src_tokens], dtype=torch.long).to(device)
    src_mask = make_src_mask(src_tensor)
    
    with torch.no_grad():
        enc_output, _ = raw_model.encoder(src_tensor, src_mask)
        tgt_tokens = [tgt_vocab.word2idx["<sos>"]]
        
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_tokens], dtype=torch.long).to(device)
            tgt_mask = make_tgt_mask(tgt_tensor)
            dec_output, _, _ = raw_model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
            logits = raw_model.generator(dec_output)
            next_token = logits[0, -1, :].argmax().item()
            tgt_tokens.append(next_token)
            if next_token == tgt_vocab.word2idx["<eos>"]:
                break
                
    return tgt_vocab.decode(tgt_tokens)


## Bước 6: Tải trọng số Model (Load Checkpoint)
Khai báo đường dẫn tới tệp checkpoint của bạn. Mặc định chúng tôi ghi nhận đường dẫn tương đối. Hãy thay đổi đường dẫn này nếu bạn chạy trên môi trường Kaggle hay các thư mục khác.


In [ ]:
# ĐƯỜNG DẪN ĐẾN FILE CHECKPOINT .PT CỦA BẠN
# Ví dụ: 'mini_transformer_best_bleu.pt' hoặc '/kaggle/input/.../mini_transformer_best_bleu.pt'
checkpoint_path = 'mini_transformer_best_bleu.pt'

# Khởi tạo mô hình Transformer với cấu hình giống lúc huấn luyện
model = Transformer(
    src_vocab_size=len(shared_vocab),
    tgt_vocab_size=len(shared_vocab),
    d_model=512,
    num_layers=6,
    num_heads=8,
    d_ff=2048,
    max_len=5000,
    dropout=0.1
)

print(f"Đang tải trọng số từ: {checkpoint_path}...")
try:
    # Đọc checkpoint
    state_dict = torch.load(checkpoint_path, map_location=device)
    
    # Loại bỏ tiền tố 'module.' nếu checkpoint được lưu từ chế độ Multi-GPU (nn.DataParallel)
    clean_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        clean_state_dict[name] = v
        
    model.load_state_dict(clean_state_dict)
    model = model.to(device)
    model.eval()
    print("🎉 Đã tải model weights thành công!")
except FileNotFoundError:
    print(f"❌ Lỗi: Không tìm thấy file checkpoint tại '{checkpoint_path}'.")
    print("👉 Vui lòng tải file weights lên hoặc kiểm tra lại đường dẫn checkpoint_path ở ô code này!")
except Exception as e:
    print(f"❌ Đã xảy ra lỗi khi load model: {e}")


## Bước 7: Dịch thử một số câu mẫu
Chạy kiểm tra bản dịch bằng cả hai thuật toán `Greedy` và `Beam Search` để so sánh.


In [ ]:
test_sentences = [
    "tôi muốn chia sẻ với các bạn một câu chuyện .",
    "đây là một thử thách lớn .",
    "chúng tôi đã học được rất nhiều thứ từ dự án này .",
    "cảm ơn các bạn rất nhiều ."
]

print("--- KẾT QUẢ DỊCH THỬ NGHIỆM ---")
for sent in test_sentences:
    print(f"Nguồn (VI): {sent}")
    # Dịch Greedy
    pred_greedy = translate(model, sent, shared_vocab, shared_vocab, mode='greedy')
    print(f"   -> Greedy: {pred_greedy}")
    # Dịch Beam Search
    pred_beam = translate(model, sent, shared_vocab, shared_vocab, mode='beam', beam_size=5)
    print(f"   -> Beam Search: {pred_beam}")
    print("-" * 50)


## Bước 8: Tạo giao diện Demo tương tác trực quan bằng Gradio
Để tạo ra một ứng dụng Demo hoàn thiện, chúng ta xây dựng giao diện Web UI thân thiện ngay trong Notebook bằng thư viện **Gradio**. Bạn có thể nhập bất kỳ câu tiếng Việt nào và xem bản dịch tiếng Anh tức thì!


In [ ]:
import gradio as gr

def translation_interface(vietnamese_sentence, mode, beam_size):
    if not vietnamese_sentence.strip():
        return "Vui lòng nhập văn bản tiếng Việt để dịch!"
    try:
        translated = translate(
            model=model, 
            src_sentence=vietnamese_sentence, 
            src_vocab=shared_vocab, 
            tgt_vocab=shared_vocab, 
            mode='beam' if mode == 'Beam Search' else 'greedy',
            beam_size=int(beam_size)
        )
        return translated
    except Exception as e:
        return f"Lỗi hệ thống: {e}"

# Thiết lập giao diện Gradio đẹp mắt với theme hiện đại
with gr.Blocks(theme=gr.themes.Soft(), css=".container { max-width: 800px; margin: auto; }") as demo:
    gr.Markdown("""
    # 🇻🇳 ➡️ 🇬🇧 Ứng dụng Demo Dịch thuật Việt - Anh
    **Mô hình sử dụng**: Mini-Transformer (huấn luyện từ đầu trên tập IWSLT 2015).  
    Hãy nhập một câu tiếng Việt bên dưới để trải nghiệm dịch thuật.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                label="Nhập câu tiếng Việt (Source Text)", 
                placeholder="Ví dụ: tôi muốn chia sẻ với các bạn một câu chuyện...", 
                lines=4
            )
            
            with gr.Row():
                mode_select = gr.Radio(
                    choices=["Beam Search", "Greedy Search"], 
                    value="Beam Search", 
                    label="Phương pháp giải mã (Decoding)"
                )
                beam_slider = gr.Slider(
                    minimum=1, 
                    maximum=10, 
                    step=1, 
                    value=5, 
                    label="Beam Size (Chỉ dùng cho Beam Search)"
                )
                
            submit_btn = gr.Button("🚀 Bắt đầu dịch", variant="primary")
            
        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label="Bản dịch tiếng Anh (English Translation)", 
                lines=6,
                interactive=False
            )
            
    # Một số câu ví dụ sẵn có để nhấn chọn nhanh
    gr.Examples(
        examples=[
            ["tôi muốn chia sẻ với các bạn một câu chuyện .", "Beam Search", 5],
            ["đây là một thử thách lớn .", "Beam Search", 5],
            ["chúng tôi đã học được rất nhiều thứ .", "Beam Search", 5],
            ["công nghệ này đang thay đổi thế giới của chúng ta .", "Beam Search", 5]
        ],
        inputs=[input_text, mode_select, beam_slider]
    )
    
    submit_btn.click(
        fn=translation_interface, 
        inputs=[input_text, mode_select, beam_slider], 
        outputs=output_text
    )

# Khởi động giao diện demo
demo.launch(share=True)
